[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/orange/notebooks/orange_protein_structures.ipynb)

# Finding a 3D structure for every target

**Orange group · Tuberculosis**

The shortlist from the previous notebook is a list of names. To judge whether a drug
could bind one of those proteins we need its shape in three dimensions. This notebook
finds one structure for every target: a measured one from the Protein Data Bank where
it is good enough, and an AlphaFold prediction where it is not.

## What you will do

- Load the 348 targets selected in the previous notebook.
- Search the Protein Data Bank for experimental structures of every target.
- Decide, target by target, whether to use the experimental structure or an AlphaFold model.
- Download one structure file per target and look inside one.
- Save the table of structures for the pocket detection notebook.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "orange"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Load the targets

We start from the shortlist made in `orange_target_selection`: essential, reliably
measured, and not equally essential in *M. smegmatis*. The group uploaded it to Drive,
and it has been copied into `data/`.

First the packages, and the one decision this notebook makes, written as a constant so it is easy to change.

In [ ]:
import os

import pandas as pd
import stylia
from scripts import structures

# Plots: slide format, Ersilia colours
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()

# Use an experimental structure only if it covers at least half of the protein
MIN_COVERAGE = 0.5
print(f"minimum coverage for a PDB structure: {MIN_COVERAGE:.0%}")

Read the shortlist. It has one row per protein, most vulnerable first.

In [ ]:
targets = pd.read_csv("data/mtb_selected_targets.csv")
print(f"{len(targets)} targets")
targets[["locus_tag", "uniprot_ac", "gene_name", "protein_name", "vi"]].head()

## 2. Experimental and predicted structures

There are two places to get a protein structure from.

The **Protein Data Bank (PDB)** holds structures that were *measured* in a laboratory,
mostly by three methods:

- **X-ray crystallography**: the protein is grown into a crystal and hit with X-rays.
- **Cryo-electron microscopy (cryo-EM)**: frozen copies of the protein are photographed
  with an electron microscope. It works well for large complexes such as the ribosome.
- **NMR**: the protein is studied in solution with magnetic fields. It works for small
  proteins only.

Two numbers tell us how good a PDB structure is for our purpose:

- **Resolution**, in ångström (Å, a tenth of a nanometre): how sharp the picture is.
  Smaller is better, and below about 3 Å the side chains are clearly visible.
- **Coverage**: the fraction of the protein that is actually in the structure. Many
  structures contain only one piece (a *domain*) of a larger protein.

**AlphaFold** is an AI model that *predicts* a structure from the sequence alone, and
the AlphaFold database has a prediction for almost every protein in UniProt. Each part
of the model comes with a confidence score, **pLDDT**, from 0 to 100. Above 90 is very
reliable, above 70 is generally right, and below 70 should not be trusted.

Let's look at InhA (`P9WGR1`), the target of isoniazid, one of the first-line TB
drugs. `pdb_best_structures` asks the PDBe service for every PDB chain that contains
the protein, best first: largest coverage, then best resolution.

In [ ]:
inha = structures.pdb_best_structures("P9WGR1")
print(f"InhA appears in {len(inha)} PDB chains")
inha.head()

The best chain covers the whole protein (coverage 1.0) at 1.4 Å, which is excellent. Here is what AlphaFold has for the same protein.

In [ ]:
structures.alphafold_entry("P9WGR1")

A mean pLDDT of 94 means AlphaFold is very confident about InhA too. But InhA is
one of the most studied proteins in TB, so it is an easy case. Most of the shortlist
has been studied far less.

## 3. Search the PDB for every target

We now ask PDBe the same question for all 348 targets and keep the best chain of each.
`best_pdb_chain` returns one row per target, and `fetch_all` sends eight requests at a
time so the whole list takes about a minute.

In [ ]:
pdb = structures.fetch_all(targets["uniprot_ac"], structures.best_pdb_chain)
has_pdb = pdb["n_pdb_chains"] > 0
print(f"{has_pdb.sum()} of {len(pdb)} targets have at least one PDB structure")
pdb[has_pdb].head()

Roughly half of the targets have been solved experimentally. Which methods were used?

In [ ]:
pdb.loc[has_pdb, "method"].value_counts()

Most are X-ray structures. Many of the cryo-EM ones are ribosomal proteins, which were
solved as part of whole-ribosome structures. Now, how much of each protein do these
structures cover?

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.hist(pdb.loc[has_pdb, "coverage"], bins=20, range=(0, 1), color=nc.orange)
ax.axvline(MIN_COVERAGE, color=nc.gray, linestyle="--")
stylia.label(ax, xlabel="Fraction of the protein in the best PDB chain",
             ylabel="Targets", title="Most PDB structures cover the whole protein")

Most cover the whole protein, but a few cover only a small piece. These are the ones to the left of the dashed line.

In [ ]:
partial = targets.merge(pdb, on="uniprot_ac")
partial = partial[partial["coverage"] < MIN_COVERAGE]
partial[["gene_name", "protein_name", "pdb_id", "coverage"]].sort_values("coverage")

Several of these are large proteins with a membrane part, such as the protein kinase
`pknB` or the arabinosyltransferase `embC`. Only the soluble part was crystallised,
so the structure misses the rest of the protein.

## 4. Choose PDB or AlphaFold

An experimental structure is *measured*, not predicted, so we prefer it. But a
structure that contains only a quarter of the protein could hide the very pocket we are
looking for. The rule is:

- **PDB** if the best chain covers at least `MIN_COVERAGE` of the protein;
- **AlphaFold** otherwise, including when there is no PDB structure at all.

In [ ]:
table = targets.merge(pdb, on="uniprot_ac")
use_pdb = table["coverage"] >= MIN_COVERAGE  # False when there is no PDB structure
table["structure_source"] = use_pdb.map({True: "PDB", False: "AlphaFold"})
table["structure_source"].value_counts()

> **Exercise:** change `MIN_COVERAGE` in section 1 to `0.8`, then to `0.2`, and run the
> notebook again. How many targets switch source? Which threshold would you defend?

## 5. Fill the gaps with AlphaFold

Next we look up the AlphaFold model of every target. We need it for the targets that
use AlphaFold, and it also tells us whether any target has no structure at all.

In [ ]:
af = structures.fetch_all(table["uniprot_ac"], structures.alphafold_entry)
table = table.merge(af, on="uniprot_ac")
no_af = table["af_url"].isna()
no_structure = no_af & table["structure_source"].eq("AlphaFold")
print(f"{no_af.sum()} targets have no AlphaFold model, "
      f"{no_structure.sum()} have no structure at all")
table.loc[no_af, ["locus_tag", "gene_name", "structure_source", "pdb_id", "coverage"]]

The one protein without an AlphaFold model has a complete PDB structure, so every
target has a structure. Now let's check how confident AlphaFold is about the models we
will actually use.

In [ ]:
chosen_af = table[table["structure_source"] == "AlphaFold"]
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.hist(chosen_af["af_plddt"], bins=25, range=(50, 100), color=nc.orange)
ax.axvline(70, color=nc.gray, linestyle="--")
stylia.label(ax, xlabel="Mean pLDDT of the model", ylabel="Targets",
             title=f"Confidence of the {len(chosen_af)} AlphaFold models we use")

Most models are confident. The few below 70 are listed here, and they deserve some caution in the next notebook.

In [ ]:
low = chosen_af[chosen_af["af_plddt"] < 70]
print(f"{len(low)} of {len(chosen_af)} AlphaFold models have a mean pLDDT below 70")
low[["gene_name", "protein_name", "af_plddt"]].sort_values("af_plddt")

> **Note:** the pLDDT here is an average over the whole protein. Even a confident model
> usually has some floppy loops or tails with low pLDDT. The next notebook checks the
> confidence of each pocket separately.

## 6. Download the structures

Now we download one file per target into `data/downloads/structures/`. Each file holds
exactly one protein chain in **PDB format**, a plain text file with one line per atom.

A PDB entry often holds more than we want: several copies of the protein, partner
proteins, water, and bound molecules. `download_structure` keeps only the chosen chain
and its protein atoms. AlphaFold models are already a single chain. Files that are
already on disk are not downloaded again. This takes a couple of minutes.

In [ ]:
table["structure_file"] = structures.download_all(table)
sizes = table["structure_file"].map(os.path.getsize)
print(f"{len(table)} files, {sizes.sum() / 1e6:.0f} MB in total, "
      f"smallest {sizes.min() / 1e3:.0f} kB")

Let's look inside the InhA file. Every atom is one line, starting with `ATOM`.

In [ ]:
inha_file = table.loc[table["gene_name"] == "inhA", "structure_file"].iloc[0]
with open(inha_file) as f:
    atoms = [line for line in f if line.startswith("ATOM")]
print(f"{inha_file}: {len(atoms):,} atoms")
print("".join(atoms[:5]))

Each line gives the atom's name (`N`, `CA`, ...), its amino acid (`MET` is
methionine), the chain (`A`), the residue number, and its x, y, z position in ångström.
The last numbers are the occupancy and the B-factor. In an X-ray structure the
B-factor says how much the atom wobbles. In an AlphaFold model, that column holds the
pLDDT instead.

## 7. Save the table

The next notebook finds pockets in these structures. It needs to know, for every target,
where its structure came from so it can download the same file again. Colab deletes
downloaded files when it disconnects, so we save the table, not the structures.

In [ ]:
table["structure_id"] = table["pdb_id"].where(use_pdb, "AF-" + table["uniprot_ac"])
COLUMNS = ["locus_tag", "uniprot_ac", "gene_name", "protein_name", "vi", "vi_lower",
           "vi_upper", "reviewed", "antibacterial", "structure_source", "structure_id",
           "pdb_id", "chain", "coverage", "resolution", "method", "n_pdb_chains",
           "af_url", "af_version", "af_plddt"]
structures_table = table[COLUMNS]
structures_table.head()

In Colab the cell below also downloads the file to your computer. Upload
`mtb_targets_structures.csv` to the group's Drive folder **Projects/OrangeTeam/Data**
so everyone works from the same list.

> **Note:** If nothing downloads, your browser may have blocked it. Look for a
> message near the address bar, allow downloads from Colab, and run the
> cell again.

In [ ]:
os.makedirs("outputs", exist_ok=True)
final_path = "outputs/mtb_targets_structures.csv"
structures_table.to_csv(final_path, index=False)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(final_path)
print(f"{len(structures_table)} targets written to {final_path}")

## Summary

- 177 of the 348 targets have at least one experimental structure in the PDB, most of
  them from X-ray crystallography and covering the whole protein.
- With a minimum coverage of 50%, 164 targets use a PDB structure and 184 use an
  AlphaFold model. Every target ends up with a structure.
- AlphaFold is confident about most of the models we use: 11 of 184 have a mean pLDDT below 70.
- Each structure is saved as a single protein chain, and the table records where it
  came from.

**Next:** `orange_pocket_detection`, which uses P2Rank to find the pockets where a drug
could bind in each of these structures.